# 📘 Tutorial 2: Batch BO for Parallel Experimentation

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

In **Tutorial 1**, we extended Bayesian Optimisation from low-dimensional, directly visualisable examples to a **genuinely higher-dimensional design space**.

We saw that, even when the BO loop itself remains structurally unchanged, the practical problem becomes harder once the number of variables increases.

In particular, we saw that high-dimensional BO requires us to think more carefully about:

- how the search space scales with dimension,
- how to initialise the optimisation sensibly,
- how to diagnose BO progress without relying on full geometry,
- and how strongly performance depends on the available evaluation budget.

That established a reusable high-dimensional BO workflow as a practical optimisation template.

In this tutorial, we take the next step:

> **what changes when experiments or simulations can be run in parallel, so that BO must choose a batch of points rather than a single next point?**

That is the focus of this notebook.

This is an important shift.

In the earlier tutorials, BO was treated mainly as a **sequential decision process**:
one surrogate, one acquisition function, one next point, one new evaluation, and then an update.

That remains the conceptual foundation.

But in many realistic research settings, that is not how evaluations are actually performed.

Instead, one BO round may correspond to:

- several reactions run together,
- multiple simulations submitted at once,
- a well plate of experimental conditions,
- a synthesis batch,
- or a set of instrument slots used in parallel.

Once that becomes possible, the BO problem changes in a meaningful way.

A batch BO problem is not just:

- the same surrogate,
- plus the same acquisition function,
- plus more points returned at once.

It also introduces new practical questions:

- how to choose **multiple points jointly** rather than one at a time,
- how to interpret progress when one BO round may contain several evaluations,
- how to compare different batch sizes fairly,
- and how to reason about the **internal geometry of the batch itself**.

So this tutorial is not mainly about introducing a completely different optimisation algorithm.

It is about learning how to adapt the BO workflow to a setting where the decision unit is no longer a single next point, but a **set of points proposed together**.

To make this concrete, the notebook uses the same style of dimension-customisable benchmark setup as the previous tutorial, but now focuses on batch selection and batch interpretation.

That lets us study several core questions:

- how to implement batch BO in BoTorch,
- how to compare different batch sizes,
- why performance by **BO round** and by **total evaluations** are not the same thing,
- how batch points are distributed within a round,
- and how the scattering or clustering behaviour of the batch can be controlled explicitly.

This is also where the notebook connects even more directly to real experimental optimisation.

In realistic research settings, the practical challenge is often not just how to choose the next best point, but how to choose the next **set of experiments worth running together**.

That is exactly the setting this tutorial is designed to approach.

---

**This tutorial is designed to shift perspective**
- from *“I can run BO in a higher-dimensional setting”*
- to *“I can build and interpret BO when each optimisation round proposes a batch of points for parallel evaluation.”*

---

**The emphasis is on developing intuition for**
- why batch BO is more than simply changing `q = 1` to `q > 1`,
- how joint batch acquisition differs from single-point selection,
- why batch size should be judged both by **BO round** and by **total evaluations**,
- how to interpret the internal geometry of a proposed batch,
- and how scattering or clustering can be encouraged through simple radius-based rules.

---

**Key ideas explored include**
- constructing a reusable **batch BO loop**,
- using **`qLogExpectedImprovement`** for joint batch selection,
- comparing BO behaviour across different batch sizes,
- distinguishing **round efficiency** from **evaluation efficiency**,
- analysing within-batch spread through pairwise-distance diagnostics,
- and introducing simple rules to encourage **scattering** or **clustering** inside the batch.

---

This tutorial serves as the bridge from:

- **custom-dimensional sequential BO** in Tutorial 1,
- to **parallel experimental decision-making** in batch Bayesian Optimisation.

In other words:

- **Tutorial 1** showed how to build and analyse BO in a genuinely higher-dimensional design space,
- and **Tutorial 2** now asks what happens when that BO workflow must choose **multiple points together** rather than one point at a time.

---

**Recommended prerequisites**
- Completion of **Tutorial 1**
- Familiarity with Gaussian Process surrogates and acquisition functions in BoTorch
- Familiarity with the standard sequential BO loop
- Comfort with best-value curves and high-dimensional BO diagnostics
- Basic awareness that real experimental workflows may evaluate several conditions in parallel

---

**Author**: Angze Li

**Last updated**: 2026-04-11

**Version**: v1.0

## 🔧 Setup

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood

from botorch.optim import optimize_acqf
from botorch.utils.sampling import draw_sobol_samples
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.acquisition import LogExpectedImprovement
from botorch.acquisition.logei import qLogExpectedImprovement

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


def style_ax(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=14)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")

## 1. Defining the problem setup and generating the initial dataset

We begin by defining the basic ingredients of the batch Bayesian Optimisation problem:

- the **input dimension**
- the **initial design size**
- the **number of BO rounds**
- the **search bounds**
- the **objective function**
- and the **initial observed dataset**

This cell therefore plays the role of a complete problem setup cell.

That is a deliberate choice.

Since the Rosenbrock objective and the initial Sobol design were already introduced in the previous tutorial, it is no longer necessary to separate every part into its own individual setup cell here.
Instead, we package the high-level configuration, objective definition, and initial data generation together so that the batch BO workflow can begin more directly.

---

### What the code does

The first part defines the main problem settings:

```python
dim = 4
n_init = 8
n_rounds = 8
seed = 7
```

This means:

- the optimisation problem has **4 input variables**
- the BO workflow begins with **8 initial observations**
- and then performs **8 batch BO rounds**

Because this is a batch BO tutorial, `n_rounds` is especially important.

A **BO round** is not the same thing as a single new evaluation.
Instead, each round will later propose an entire **batch** of candidate points to be evaluated together.

So this notebook is structured around **parallel decision rounds**, not purely sequential one-point updates.

---

### What `seed = 7` is doing

The line

```python
torch.manual_seed(seed)
```

sets the PyTorch random seed.

This is used so that random or quasi-random operations in the notebook become reproducible, including:

- the initial Sobol design
- and other stochastic components of the BO workflow

So `seed = 7` does not have any special mathematical meaning by itself.
It is simply a fixed integer chosen to make the notebook reproducible.

If the same code is rerun with the same seed, the initial design and BO trajectory should be consistent up to ordinary numerical reproducibility.

---

### What the bounds mean

The code then defines:

```python
lower_bound = -2.0
upper_bound = 2.0
```

and creates the BoTorch bounds tensor:

```python
bounds = torch.tensor(
    [[lower_bound] * dim, [upper_bound] * dim],
    dtype=torch.double,
)
```

Since `dim = 4`, this means every input variable is constrained to lie in the interval:

```python
[-2, 2]
```

So the full search domain is:

```python
[-2, 2]^4
```

In BoTorch, bounds are stored in shape:

```python
(2, d)
```

where:

- the first row contains the lower bounds
- the second row contains the upper bounds

So the printed shape `(2, 4)` confirms that the search space has been defined correctly for a 4-dimensional problem.

---

### Why `variable_names` is defined

The line

```python
variable_names = [f"x{i + 1}" for i in range(dim)]
```

creates simple coordinate labels:

```python
["x1", "x2", "x3", "x4"]
```

These are useful later for:

- coordinate-trace plots
- batch-candidate visualisation
- and summary tables of the best design found

In a real application, these would usually be replaced with meaningful names such as:

- temperature
- pH
- catalyst loading
- reaction time

but generic coordinate names are a clean default for a tutorial notebook.

---

### Why Rosenbrock is used again

The next part defines the shifted Rosenbrock objective:

```python
def objective_rosenbrock(X, center=None):
    ...
```

We use Rosenbrock again here for continuity with the previous tutorial.

That is useful because it means this tutorial can focus on the **new batch BO ideas** without also introducing a completely new benchmark at the same time.

Rosenbrock is a good choice because it is:

- dimension-customisable
- smooth but nontrivial
- harder than a simple quadratic
- and still interpretable enough for BO tutorials

So it provides a useful test case for studying how batch BO behaves when the objective is not completely trivial.

---

### The mathematical meaning of the objective

The Rosenbrock function is built from coupled terms of the form:

$$
100\left(z_{i+1} - z_i^2\right)^2 + (1 - z_i)^2
$$

summed across neighbouring coordinates.

This creates a narrow curved valley leading to the optimum.

So the optimisation difficulty comes less from heavy multimodality and more from:

- anisotropy
- variable coupling
- and the need to make coordinated progress across dimensions

That makes it a useful benchmark for both sequential and batch BO.

---

### Why the optimum is shifted

The code defines:

```python
Z = X - center + 1.0
```

which means the standard Rosenbrock minimum is shifted so that the optimum lies at `center` rather than at a trivial coordinate like all ones.

If `center` is not provided, it is defined automatically as:

```python
torch.linspace(-1.5, 1.5, d, ...)
```

So the optimum lies inside the allowed search region but not at an especially symmetric or trivial point.

This makes the benchmark slightly more realistic and less artificial.

---

### Why the seed is reset before drawing the initial design

The code uses:

```python
torch.manual_seed(seed)
```

again just before creating the Sobol design.

This ensures that the initial dataset is generated from the intended reproducible starting state, regardless of what may have happened earlier in the notebook.

So this second seed reset is a way of making the initial design generation especially controlled and repeatable.

---

### Why Sobol is used for the initial design

The initial dataset is created using:

```python
train_X = draw_sobol_samples(bounds=bounds, n=1, q=n_init).squeeze(0)
train_Y = objective_rosenbrock(train_X)
```

This means:

- `train_X` contains `n_init = 8` initial points in the 4D search domain
- `train_Y` contains the corresponding Rosenbrock objective values

A Sobol design is used because it gives a more **space-filling** initial coverage than naive random sampling.

That is especially important in higher-dimensional BO, where poor initial coverage can make the early GP surrogate much less informative.

So Sobol is a strong default for the initial design.

---

### What the printed outputs tell us

The final print statements confirm:

- the input shape of the initial design
- the output shape of the initial objective values
- and the best objective value among the initial points

So this gives us a compact summary of the starting dataset before any BO rounds have begun.

This best initial value is also useful because later BO performance plots can be interpreted relative to this starting point.

---

### Why this cell is enough as a combined setup cell

In the previous tutorial, it made sense to separate:

- problem setup
- objective definition
- and initial design generation

because those ideas were being introduced carefully one by one.

Here, the goal is different.

Since those ingredients are already familiar, combining them into one cell makes the notebook more efficient and keeps the focus on what is actually new in this tutorial:

- joint batch acquisition
- parallel candidate selection
- and the comparison of different batch sizes

So this combined cell is a sensible design choice for Tutorial 2.

---

### Key takeaway

This cell defines the full batch BO problem setup in one place.

It specifies:

- a 4-dimensional design space
- an 8-point initial Sobol design
- 8 BO rounds
- a shifted Rosenbrock objective
- and the initial observed dataset

Together, these ingredients create the starting point for the batch BO experiments studied throughout the rest of the notebook.

In [ ]:
dim = 4
n_init = 8
n_rounds = 8
seed = 7

torch.manual_seed(seed)

lower_bound = -2.0
upper_bound = 2.0

bounds = torch.tensor(
    [[lower_bound] * dim, [upper_bound] * dim],
    dtype=torch.double,
)

variable_names = [f"x{i + 1}" for i in range(dim)]

print("Input dimension:", dim)
print("Initial design size:", n_init)
print("BO rounds:", n_rounds)
print("Bounds shape:", tuple(bounds.shape))
print("Variable names:", variable_names)

def objective_rosenbrock(X, center=None):
    X = X.to(dtype=torch.double)
    d = X.shape[-1]

    if center is None:
        center = torch.linspace(-1.5, 1.5, d, dtype=torch.double, device=X.device)

    if center.ndim == 1:
        center = center.unsqueeze(0)

    Z = X - center + 1.0
    y = torch.sum(
        100.0 * (Z[..., 1:] - Z[..., :-1] ** 2) ** 2
        + (1.0 - Z[..., :-1]) ** 2,
        dim=-1,
    )

    return y.unsqueeze(-1)

torch.manual_seed(seed)

train_X = draw_sobol_samples(bounds=bounds, n=1, q=n_init).squeeze(0)
train_Y = objective_rosenbrock(train_X)

print("train_X shape:", tuple(train_X.shape))
print("train_Y shape:", tuple(train_Y.shape))
print("Initial best observed value:", torch.min(train_Y).item())

## 2. Defining a reusable batch BO loop

With the problem setup and initial dataset now in place, we can define the main computational engine of this tutorial: a **reusable batch Bayesian Optimisation loop**.

This is the key transition from the previous tutorial.

In the earlier sequential BO setting, each optimisation step selected **one** new point at a time.
Here, the BO loop is modified so that each optimisation round can propose an entire **batch** of candidate points to be evaluated in parallel.

So the purpose of this function is to implement the standard BO workflow in a form suitable for **parallel experimentation**.

---

### What the function does

The function

```python
def run_batch_bo_loop(...):
```

takes:

- an initial observed dataset
- a black-box objective function
- the search bounds
- a chosen batch size
- and a number of BO rounds

and then runs a batch BO workflow for `n_rounds` rounds.

At a high level, each round does the following:

1. convert the minimisation problem into a maximisation-compatible form
2. fit a Gaussian Process surrogate
3. build a **batch acquisition function**
4. jointly optimise that acquisition function to select `batch_size` points
5. evaluate all points in the batch
6. append the full batch to the dataset
7. store the BO state in `history`

So this function is the batch analogue of the sequential BO loop from the previous tutorial.

---

### Why `train_X` and `train_Y` are cloned first

The lines

```python
train_X = train_X_init.clone()
train_Y = train_Y_init.clone()
```

make local working copies of the initial dataset.

This is the same good practice as before: it allows the function to update its internal dataset without mutating the original input tensors outside the function.

So the BO history is built on internal copies, while the original initial design remains unchanged.

---

### Why minimisation is converted into maximisation again

The code begins each round with:

```python
train_Y_bo = -train_Y
y_best_bo = train_Y_bo.max().item()
```

This is done for the same reason as in standard BoTorch BO.

The Rosenbrock problem is a **minimisation** problem, but the acquisition function machinery is most naturally written in **maximisation** form.

So the objective values are negated, turning:

- minimising `train_Y`
- into maximising `train_Y_bo = -train_Y`

This is a standard and important practical step.

---

## What has changed to allow batch BO

The most important change in this function is that the BO loop is no longer selecting a single next point.

Instead, it is selecting a **set of points jointly**.

That required several specific modifications relative to the sequential BO loop.

---

### 1. The acquisition function is now a batch acquisition function

In the sequential BO case, we used an acquisition function designed for one-point decisions.

Here, we instead define:

```python
qlogei = qLogExpectedImprovement(
    model=gp,
    best_f=y_best_bo,
    sampler=sampler,
)
```

The crucial change is the use of:

```python
qLogExpectedImprovement
```

rather than the single-point LogEI used in the sequential setting.

This is important because batch BO is not simply “pick the best point and repeat it several times.”

Instead, the acquisition function must score the value of a **set of candidate points considered together**.

So this is one of the main conceptual changes that enables batch BO.

---

### 2. A Monte Carlo sampler is now needed

The code defines:

```python
sampler = SobolQMCNormalSampler(sample_shape=torch.Size([mc_samples]))
```

This is another important difference from the single-point case.

For batch BO, the acquisition function is typically evaluated through **Monte Carlo estimation**, because the joint value of multiple candidate points is harder to compute analytically.

So the sampler is used to approximate the batch acquisition function under the GP posterior.

That is why the function now includes the extra parameter:

```python
mc_samples=128
```

This controls how many Monte Carlo samples are used when estimating `qLogExpectedImprovement`.

So another key change for batch BO is that the acquisition step becomes both:

- **joint**
- and **Monte Carlo based**

---

### 3. The acquisition optimisation now selects multiple points at once

The most visible code change is here:

```python
candidates, acq_value = optimize_acqf(
    acq_function=qlogei,
    bounds=bounds,
    q=batch_size,
    num_restarts=num_restarts,
    raw_samples=raw_samples,
)
```

The parameter

```python
q=batch_size
```

is what tells BoTorch to optimise over a **batch of points** rather than a single point.

This means the returned `candidates` tensor has shape approximately:

```python
(batch_size, d)
```

rather than `(1, d)`.

So instead of selecting one next evaluation, the BO loop now selects a full batch of `batch_size` candidates in one joint optimisation step.

That is the central mechanical change that makes this a batch BO loop.

---

### 4. The dataset update now appends a whole batch

In the sequential setting, one new observation was appended per BO step.

Here, the update is:

```python
X_new = candidates
Y_new = objective_fn(X_new)

train_X = torch.cat([train_X, X_new], dim=0)
train_Y = torch.cat([train_Y, Y_new], dim=0)
```

So each BO round now adds **multiple new observations** at once.

This is another essential change:

- one round no longer means one new evaluation
- one round means one **parallel batch of evaluations**

That changes both the dataset growth and the interpretation of BO progress.

---

### 5. The loop is now organised around BO rounds rather than single evaluations

The parameter

```python
n_rounds
```

is used rather than something like `n_steps` that implicitly suggests one new point per iteration.

This naming matters conceptually.

A **round** is now a parallel decision stage in which several experiments or simulations may be launched together.

So the unit of progress in this tutorial is not “one new candidate,” but “one new batch.”

That makes the interpretation much more aligned with real parallel experimentation.

---

### Why batch BO is not just “changing `q`”

Although `q=batch_size` is the key mechanical switch inside `optimize_acqf`, the change is deeper than that.

Batch BO also changes:

- the acquisition function family
- the need for Monte Carlo estimation
- the structure of the returned candidate tensor
- the dataset update step
- and the interpretation of one BO round

So this function is not just a sequential loop with a different tensor shape.
It is a genuinely different BO decision structure.

---

### Why `history` is still stored

As in the previous tutorial, the function stores a full `history` list.

This is important because, once BO is run in batches, we want to analyse not only:

- the best value found so far
- but also:
  - the coordinates of proposed batch points
  - the diversity of each batch
  - the learned GP lengthscales
  - and the evolution of the best-so-far design

So the stored history remains essential for later diagnostic plots.

---

### Why lengthscale and noise are still stored

The code extracts:

```python
lengthscale = gp.covar_module.lengthscale.detach().cpu().reshape(-1).numpy()
noise = gp.likelihood.noise.detach().cpu().item()
```

These are the same GP hyperparameter diagnostics used in earlier tutorials.

Even though the acquisition behaviour is now batch-based, the surrogate is still a GP, so it is still useful to inspect how the model’s internal view of the objective evolves over rounds.

So these stored quantities remain valuable for interpretation.

---

### Why `best_x` and `best_y` are stored again

The code also keeps track of:

```python
best_idx = torch.argmin(train_Y)
best_x = train_X[best_idx].clone()
best_y = train_Y[best_idx].clone()
```

This remains useful because, even in a batch BO setting, one of the most important practical questions is still:

> **What is the best solution found so far?**

So storing the current best design and best objective value allows later plots and summary tables to focus on optimisation progress, not only raw candidate generation.

---

### One subtle point about the final stored state

The loop stores a BO state for every round index from:

```python
0
```

to

```python
n_rounds
```

but only appends new observations when:

```python
round_idx < n_rounds
```

So the final stored history state includes a final proposed batch of `candidates` that has been generated but not yet evaluated.

This is useful for inspection, but it does mean that some later analyses should use:

```python
history[:-1]
```

if they want only batches that were actually evaluated.

That is an important detail to remember when plotting within-batch diversity or candidate coordinates.

---

### Why this function matters for the rest of the notebook

This function is the main engine of the tutorial.

It is what makes it possible to compare:

- different batch sizes
- performance by BO round
- performance by total evaluations
- and batch diversity effects

without rewriting the BO workflow each time.

So this function is not just one example implementation.
It is the reusable core of the batch BO experiments that follow.

---

### Key takeaway

This cell defines a reusable **batch BO loop** in which each optimisation round proposes and evaluates multiple points together.

The main changes that allow batch BO are:

- replacing the single-point acquisition with **`qLogExpectedImprovement`**
- introducing a **Monte Carlo sampler** with `SobolQMCNormalSampler`
- optimising the acquisition with **`q=batch_size`**
- updating the dataset with a full batch of new observations
- and interpreting optimisation progress in terms of **BO rounds** rather than single evaluations

So this function is the batch counterpart of the sequential BO loop from the previous tutorial.

In [ ]:
def run_batch_bo_loop(
        train_X_init,
        train_Y_init,
        objective_fn,
        bounds,
        batch_size=2,
        n_rounds=8,
        num_restarts=20,
        raw_samples=256,
        mc_samples=128,
):
    train_X = train_X_init.clone()
    train_Y = train_Y_init.clone()

    history = []

    for round_idx in range(n_rounds + 1):
        train_Y_bo = -train_Y
        y_best_bo = train_Y_bo.max().item()

        gp = SingleTaskGP(
            train_X=train_X,
            train_Y=train_Y_bo,
            input_transform=Normalize(d=train_X.shape[-1]),
            outcome_transform=Standardize(m=1),
        )
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll(mll)
        gp.eval()

        sampler = SobolQMCNormalSampler(sample_shape=torch.Size([mc_samples]))
        qlogei = qLogExpectedImprovement(
            model=gp,
            best_f=y_best_bo,
            sampler=sampler,
        )

        candidates, acq_value = optimize_acqf(
            acq_function=qlogei,
            bounds=bounds,
            q=batch_size,
            num_restarts=num_restarts,
            raw_samples=raw_samples,
        )

        lengthscale = gp.covar_module.lengthscale.detach().cpu().reshape(-1).numpy()
        noise = gp.likelihood.noise.detach().cpu().item()

        best_idx = torch.argmin(train_Y)
        best_x = train_X[best_idx].clone()
        best_y = train_Y[best_idx].clone()

        history.append({
            "round": round_idx,
            "train_X": train_X.clone(),
            "train_Y": train_Y.clone(),
            "candidates": candidates.clone(),
            "acq_value": acq_value.clone(),
            "batch_size": batch_size,
            "best_x": best_x.clone(),
            "best_y": best_y.clone(),
            "lengthscale": lengthscale.copy(),
            "noise": noise,
        })

        if round_idx < n_rounds:
            X_new = candidates
            Y_new = objective_fn(X_new)

            train_X = torch.cat([train_X, X_new], dim=0)
            train_Y = torch.cat([train_Y, Y_new], dim=0)

    return history

## 3. Running one example batch BO loop

With the reusable batch BO engine now defined, we can run one concrete example of batch Bayesian Optimisation on the current 4-dimensional Rosenbrock problem.

In this example, we choose:

- `batch_size = 4`
- `n_rounds = 8`

So each BO round proposes **4 candidate points jointly**, and the optimisation is allowed to proceed for **8 parallel decision rounds**.

This gives a concrete worked example before we later compare different batch sizes more systematically.

---

### What the code does

The main function call is:

```python
history_batch = run_batch_bo_loop(...)
```

This runs the batch BO loop using:

- the initial Sobol dataset
- the shifted Rosenbrock objective
- a batch size of 4
- and 8 BO rounds

The resulting object `history_batch` stores the full batch BO trajectory, including:

- the observed dataset at each stored state
- the candidate batch proposed at that state
- the current best point and best value
- and the learned GP hyperparameters

So `history_batch` is the main object we will use to analyse the behaviour of this particular batch BO run.

---

### Why `n_rounds = 8` but the number of stored BO states is 9

A subtle but important point is that the code prints:

```python
Number of stored BO states: 9
```

even though we specified:

```python
n_rounds = 8
```

This is expected.

The reason is that the loop is written as:

```python
for round_idx in range(n_rounds + 1):
```

So the function stores:

- the **initial state before any batch update**
- plus one stored state for each of the 8 BO rounds

That gives:

```python
8 + 1 = 9
```

stored BO states in total.

So the extra stored state is not an error.
It corresponds to the fact that the history includes the starting point of the optimisation, not only the updated states.

---

### Why the final stored batch is proposed but not evaluated

Another important subtlety is that the final stored state includes a batch of:

```python
"candidates"
```

that has been **proposed** by the acquisition function but **not evaluated yet**.

This happens because the loop stores the state first, and only evaluates the batch when:

```python
if round_idx < n_rounds:
```

So when `round_idx == n_rounds`, the function still fits the GP and proposes one more candidate batch, but it does **not** append that final batch to the observed dataset.

That means:

- the final stored state is useful for inspection
- but the final candidate batch has not actually been run on the objective

So the history contains:

- 9 stored BO states
- but only 8 evaluated batch updates

This is why later diagnostic plots that focus on **evaluated candidate batches** often use:

```python
history_batch[:-1]
```

to exclude that final unevaluated proposal.

---

### Why the final number of observations is 40

The printed output shows:

```python
Final number of observations: 40
```

This can be understood directly from the initial design and the batch updates.

We started with:

- `n_init = 8` initial observations

Then each of the 8 BO rounds adds:

- `batch_size = 4` new evaluations

So the final total is:

$$
8 + 8 \times 4 = 40
$$

which matches the printed result exactly.

So this is a useful consistency check that the batch BO loop is updating the dataset as intended.

---

### What the final best observed value tells us

The line

```python
print("Final best observed value:", float(torch.min(history_batch[-1]["train_Y"])))
```

reports the best objective value found after all **evaluated** BO rounds.

This is one of the most practically important outputs, because in a real optimisation setting this is exactly the kind of summary we care about:

> **what is the best condition found so far after spending this evaluation budget?**

So even though the final stored history entry also includes one more proposed batch, the reported best value still refers to the actually observed dataset available at that point.

---

### Why the last batch candidate shape is `(4, 4)`

The final print statement gives:

```python
Last batch candidate shape: (4, 4)
```

This is also exactly what we should expect.

The shape means:

- `4` candidate points in the batch
- each point lives in a `4`-dimensional design space

So this confirms that the batch acquisition step is returning the right kind of object: a full batch of four 4D candidate designs.

---

### Why this cell matters

This cell is the first full demonstration that the batch BO loop works as intended.

It shows that:

- the optimisation can be run end-to-end
- the dataset grows by full batches
- the history object stores both optimisation progress and candidate proposals
- and the bookkeeping of rounds, observations, and batch shape is internally consistent

So this cell serves as the concrete worked example that the rest of the tutorial builds on.

---

### Key takeaway

This cell runs one example batch BO workflow with:

- `batch_size = 4`
- and `n_rounds = 8`

The reason the output shows **9 stored BO states** is that the history includes:

- the initial pre-update state
- plus one stored state for each of the 8 BO rounds

The final stored state also contains a batch that has been **proposed but not evaluated**, which is why later analyses that focus only on evaluated candidate batches often use `history_batch[:-1]`.

In [ ]:
history_batch = run_batch_bo_loop(
    train_X_init=train_X,
    train_Y_init=train_Y,
    objective_fn=objective_rosenbrock,
    bounds=bounds,
    batch_size=4,
    n_rounds=8,
    num_restarts=20,
    raw_samples=256,
    mc_samples=128,
)

print("Number of stored BO states:", len(history_batch))
print("Final number of observations:", history_batch[-1]["train_X"].shape[0])
print("Final best observed value:", float(torch.min(history_batch[-1]["train_Y"])))
print("Last batch candidate shape:", tuple(history_batch[-1]["candidates"].shape))

## 4. Tracking the best observed value over batch BO rounds

Once the batch BO loop has been run, the most direct performance diagnostic is the **best observed objective value so far**.

This is one of the most useful quantities to track in practice, because even in a parallel experimentation setting, the central optimisation question remains:

> **How good is the best condition found so far?**

That is exactly what this plot shows.

---

### What the code does

The first line extracts the best observed value from each stored BO state:

```python
best_vals_batch = [float(torch.min(h["train_Y"])) for h in history_batch]
```

Each entry in `history_batch` contains the full observed dataset available at that stored state.
So for each round, the code computes:

```python
min(train_Y)
```

which is the best objective value found **so far**.

Because this is a minimisation problem, lower values are better.

So `best_vals_batch` is a cumulative best-value trajectory across the batch BO run.

The figure then plots that trajectory against BO round.

---

### How to read the plot

The horizontal axis is:

- **BO round**

The vertical axis is:

- **best observed value**

So each point answers the question:

> *After this many batch BO rounds, what is the lowest objective value found anywhere in the observed dataset so far?*

Since this is a cumulative best-value curve:

- it can stay flat for several rounds
- and it only drops when a newly evaluated batch contains at least one point that improves on the current best solution

So the plot should not be expected to decrease smoothly at every round.

Plateaus are completely normal.

---

### Why the curve often improves in jumps rather than smoothly

This kind of stepwise behaviour is typical in Bayesian Optimisation.

At a given round, the batch acquisition function proposes a **set of candidate points** that look promising under the current surrogate model.
But that does **not** mean every batch will immediately contain a better point than the current best one.

So in many runs, the curve will show a pattern like:

- an early improvement from the initial design
- one or more flat stretches where the current best value is not beaten
- then a sudden drop when the optimiser discovers a genuinely better region
- followed by more plateaus and occasional further improvements

That is a very natural BO behaviour.

---

### Why a flat section is not a problem

A flat section in this figure does not mean batch BO has failed.

It simply means that the most recent batch or batches did not improve the best solution found so far.

That can happen for several reasons:

- the surrogate may still be uncertain about where the strongest region lies
- the batch may be probing multiple plausible areas without immediately improving the best value
- or the optimiser may be refining its model before finding a stronger basin later

So the best-value curve should be interpreted as a summary of **optimisation progress**, not as a guarantee of improvement at every round.

---

### What this plot means specifically in the batch setting

In this tutorial, one round corresponds to a **batch** of evaluations rather than a single new point.

That means each downward jump in the curve reflects the fact that **at least one point in that batch** improved the current best value.

So the plot is not tracking the quality of each individual point separately.
It is tracking the best outcome available after each round of parallel experimentation.

This is exactly the right way to think about batch BO in practice.

If a lab can run several experiments in parallel, what matters operationally after each round is:

> **what is the best result we now have in hand?**

That is the quantity this curve records.

---

### Why this diagnostic is useful even if the seed changes

The exact numerical shape of the curve may change if:

- a different random seed is used
- the initial Sobol design changes
- or the acquisition optimiser follows a slightly different trajectory

That is expected.

So the purpose of this figure is not to highlight one exact numerical path, but to illustrate the **general behaviour** of batch BO.

Across different runs, the same qualitative interpretation usually still holds:

- the best observed value tends to improve in stages
- improvements often come in discrete jumps
- and some rounds may add useful information without immediately lowering the current best value

So this plot is best read as a picture of the **general trend** of batch BO rather than as a seed-specific deterministic signature.

---

### Why this is an important batch BO diagnostic

In higher-dimensional settings, full visualisation of the objective and surrogate is no longer possible.

So plots like this become especially important.

They provide a compact summary of whether the optimisation is:

- making progress
- plateauing
- or continuing to find substantially better regions over time

That makes this one of the most useful performance plots in the notebook.

---

### What this plot can and cannot tell us

This figure tells us:

- whether the best found value is improving
- when major improvements happen
- and how optimisation progress is distributed across BO rounds

But it does **not** tell us:

- how diverse the proposed batches are
- which coordinates are changing most strongly
- or whether improvement is being driven by exploration or local refinement

That is why the later diagnostics still matter.

This curve is the most direct optimisation summary, but it is not the whole story.

---

### Key takeaway

This figure tracks the **best observed objective value so far** across the batch BO run.

The general trend to look for is:

- improvement in stages rather than smoothly,
- plateaus where recent batches do not beat the current best value,
- and occasional downward jumps when a newly evaluated batch discovers a stronger point.

That is a realistic and important pattern in batch Bayesian Optimisation, where progress is made through repeated rounds of parallel candidate selection rather than single-point updates.

In [ ]:
best_vals_batch = [float(torch.min(h["train_Y"])) for h in history_batch]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(best_vals_batch)), best_vals_batch, "-o", linewidth=2.5)
ax.set_title("Best observed value over BO rounds (batch size = 4)", fontsize=18, fontweight="bold")
ax.set_xlabel("BO round", fontsize=22, fontweight="bold")
ax.set_ylabel("Best observed value", fontsize=22, fontweight="bold")
style_ax(ax)
plt.tight_layout()
plt.show()

## 5. Visualising the coordinates proposed in each BO batch

The best-observed-value curve tells us **whether** batch BO is improving, but it does not tell us **what kind of candidate sets the optimiser is actually proposing**.

That is why this figure is useful.

Instead of summarising only the best value found so far, it visualises the **raw coordinates of the batch candidates** proposed at each BO round.

In a batch BO setting, this matters a lot, because one optimisation round no longer means selecting a single point.
It means selecting a **set of points jointly**.

So to understand batch BO properly, we need to look not only at the best value trajectory, but also at the internal structure of the proposed batches.

---

### What the code does

The first line extracts the candidate batches from the stored BO history:

```python
candidate_batches = [h["candidates"] for h in history_batch[:-1]]
```

The slice `[:-1]` is important.

As discussed earlier, the final history entry contains a batch that has been **proposed but not evaluated**.
So here we restrict attention to the batches that were actually part of completed BO rounds.

The figure then creates a 2×2 grid of subplots, one for each coordinate:

- `x1`
- `x2`
- `x3`
- `x4`

For each coordinate `j`, the code plots all values of that coordinate from the proposed batch at each BO round.

So if `batch_size = 4`, then each round contributes **4 points** to each subplot.

This means the reader can see, for each variable separately:

> how the optimiser is distributing the coordinates of the batch points across BO rounds.

---

### How to read the figure

Each panel corresponds to one input variable.

- the horizontal axis is **BO round**
- the vertical axis is the value of one coordinate, such as `x1` or `x3`

At a given BO round, the multiple scatter points stacked vertically represent the values of that coordinate across all points in the batch.

So one vertical stack is the coordinate-level “fingerprint” of the entire proposed batch at that round.

This means the figure is not showing one optimisation trajectory.
It is showing, coordinate by coordinate, how the **whole batch proposal** is structured at each round.

---

### Why this is important in batch BO

This figure is important because batch BO is fundamentally different from single-point BO.

In sequential BO, we only need to ask:

> where is the next point?

But in batch BO, we need to ask:

> what set of points is being proposed together?

That is a much richer decision problem.

A batch can be:

- tightly clustered
- moderately spread out
- concentrated in one region in some coordinates
- and diverse in others

So without a plot like this, it is hard to see whether batch BO is actually behaving like a **joint multi-point decision process** rather than just a repeated single-point heuristic.

---

### What kinds of patterns to look for

This figure is useful because it can reveal several qualitatively different behaviours.

#### 1. Tight clustering within a round
If the points in one vertical stack are close together, that means the batch is concentrating on a relatively narrow region in that coordinate.

This may suggest stronger local refinement or exploitation.

#### 2. Broad spread within a round
If the points in one stack are more spread out, that means the batch is allocating diversity in that coordinate.

This may reflect broader exploration or uncertainty reduction.

#### 3. Changing coordinate ranges across rounds
If the locations of the stacks shift over BO rounds, that means the optimiser is moving the search focus over time.

So the figure can also show whether BO is stabilising around a region or continuing to probe new parts of the space.

---

### Why the figure should be interpreted qualitatively

The exact pattern of points in this plot will depend on:

- the random seed
- the initial Sobol design
- the fitted surrogate
- and the resulting acquisition-optimisation trajectory

So the purpose of the figure is not to claim that one exact visual pattern is universal.

Instead, it is meant to illustrate the general idea that:

> in batch BO, each round produces a structured set of coordinates, not just a single recommendation.

That is the key insight.

So even if a different seed produces slightly different stacks and spreads, the same type of interpretation still applies.

---

### Why this figure complements the best-value plot

The earlier best-value plot tells us:

- when BO improved
- when it plateaued
- and how much progress it made across rounds

But it does not tell us how the optimiser was allocating its batch decisions internally.

This figure fills that gap.

It helps us see whether the batch points are:

- very similar to one another
- moderately diverse
- or shifting between regions over rounds

So the two plots answer different questions:

- the best-value plot asks **how well is batch BO doing?**
- this coordinate plot asks **what kind of batches is batch BO proposing?**

That makes them complementary.

---

### Why this matters for real experimentation

In real lab or simulation workflows, a batch is not just an abstract mathematical object.

It may correspond to:

- several experiments run in parallel
- a well plate of conditions
- multiple simulation jobs submitted together
- or several instrument slots used in the same round

In those settings, it matters whether the batch points are:

- almost duplicates
- deliberately diverse
- or concentrated around one promising region

So this figure helps connect the BO output back to the real meaning of a batch in practice.

It makes the proposed candidate sets more interpretable as **experimental plans** rather than just tensors returned by BoTorch.

---

### Key takeaway

This figure visualises the **coordinate structure of the proposed batch points** at each BO round.

That is important because, in batch BO, one decision round produces a **set of candidate designs**, not just one next point.

So this plot helps reveal how the optimiser is distributing the batch across the search space, coordinate by coordinate, and gives a much clearer picture of what batch BO is actually proposing internally.

In [ ]:
candidate_batches = [h["candidates"] for h in history_batch[:-1]]

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
axes = np.array(axes).reshape(-1)

for j in range(dim):
    ax = axes[j]

    for round_idx, batch in enumerate(candidate_batches):
        x_positions = np.full(batch.shape[0], round_idx)
        ax.scatter(
            x_positions,
            batch[:, j].numpy(),
            s=25,
            zorder=3,
        )
    for ax in axes[dim:]:
        ax.set_visible(False)

    ax.set_ylabel(variable_names[j], fontsize=14, fontweight="bold")
    style_ax(ax)

fig.suptitle("Candidate coordinates proposed in each BO batch", fontsize=18, fontweight="bold")
axes[2].set_xlabel("BO round", fontsize=18, fontweight="bold")
axes[3].set_xlabel("BO round", fontsize=18, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Running batch BO under different batch sizes

So far, the notebook has shown one concrete example of batch Bayesian Optimisation with a fixed batch size.

The next natural question is:

> **How does BO behaviour change when the number of points proposed per batch is varied?**

That is what this cell is designed to study.

Instead of changing the objective, the bounds, or the BO modelling setup, we keep the optimisation problem fixed and vary only the **batch size**:

- `q = 1`
- `q = 2`
- `q = 4`
- `q = 6`

This lets us compare how the same BO workflow behaves when each optimisation round proposes:

- one point
- a small batch
- a medium batch
- or a larger batch

So the comparison here is fundamentally about the effect of **parallelism level** on BO behaviour.

---

### What the code does

The line

```python
batch_sizes_to_compare = [1, 2, 4, 6]
```

defines the batch sizes that will be compared.

An empty dictionary is then created:

```python
batch_histories = {}
```

This will store one full BO history for each batch size.

Then, for each batch size `q`, the code:

1. resets the random seed
2. regenerates the same-size initial Sobol design
3. evaluates the Rosenbrock objective on that initial design
4. runs the batch BO loop with `batch_size=q`
5. stores the full history in `batch_histories[q]`

So at the end of the cell, `batch_histories` contains four complete BO runs, one for each chosen batch size.

---

### Why the random seed is reset inside the loop

Inside the loop, the code calls:

```python
torch.manual_seed(seed)
```

before regenerating the initial design.

This is important because we want the comparison across batch sizes to be as fair and reproducible as possible.

Resetting the seed ensures that each run starts from a controlled random state rather than inheriting randomness from previous iterations of the loop.

That does **not** make the runs identical in every respect, because changing the batch size changes the entire BO trajectory.
But it does make the comparison more consistent by ensuring that the initial Sobol design is generated in a reproducible way for each batch-size condition.

So the differences we later observe are driven much more by the changed batch size than by uncontrolled randomness in the starting point.

---

### Why the initial dataset is regenerated for each batch size

For each `q`, the code builds:

```python
train_X_q = draw_sobol_samples(bounds=bounds, n=1, q=n_init).squeeze(0)
train_Y_q = objective_rosenbrock(train_X_q)
```

This means every batch-size experiment starts from its own initial dataset, generated in the same general way.

This is a sensible design because it keeps the workflow self-contained for each batch-size condition.
Each run is a full BO experiment from initial design to final history.

Because the seed is reset each time, the starting datasets are comparable across runs.

---

### What exactly is being compared

The key thing being changed here is:

```python
batch_size=q
```

inside the call to `run_batch_bo_loop(...)`.

So the main comparison is:

- how BO behaves when each round selects **one point**
- versus **two points**
- versus **four points**
- versus **six points**

This matters because the batch size changes the structure of the decision problem.

A larger batch means:

- more points are proposed jointly in each round
- more evaluations are collected before the next surrogate update
- and the optimisation is making broader parallel decisions rather than finer sequential ones

So this is not merely a comparison of tensor shapes.
It is a comparison of different **parallel experimentation strategies**.

---

### Why `q = 1` is still included here

Even though this tutorial is focused on batch BO rather than a recap of sequential BO, it is still useful to include:

```python
q = 1
```

as one of the comparison points.

Here, `q = 1` is not being used as a separate tutorial section or conceptual recap.
It is simply the smallest possible batch size and therefore a natural baseline in the batch-size comparison.

This gives a reference point for understanding what changes as the batch becomes larger.

So `q = 1` is included as a comparison condition, not as a detour away from batch BO.

---

### Why this comparison is important

This cell sets up one of the central practical questions of the whole tutorial:

> **If I can run more experiments in parallel, how does that change the optimisation behaviour?**

That question matters in real applications because batch size is often determined by experimental or computational resources.

For example, the feasible batch size might depend on:

- how many reactions can be run at once
- how many wells fit on a plate
- how many simulations can be launched in parallel
- or how many instrument slots are available per round

So comparing different values of `q` is not just a mathematical exercise.
It is directly related to real optimisation workflows.

---

### What we are *not* comparing here

This comparison keeps all of the following fixed:

- the objective function
- the input dimension
- the bounds
- the initial design size
- the BO model structure
- the acquisition-optimisation settings
- and the number of BO rounds

So the point is **not** to compare different BO algorithms in general.

The point is to isolate the effect of changing only the **batch size**.

That makes the later plots much easier to interpret.

---

### What this prepares us to examine next

Once these histories are stored, we can compare batch-size behaviour in several ways, including:

- best observed value by **BO round**
- best observed value by **total evaluations**
- the diversity of the proposed batch points
- and the final best designs found under each batch size

Those later diagnostics are important because batch size affects more than one notion of performance.

---

### Key takeaway

This cell runs the same BO workflow under four different batch sizes:

- `q = 1`
- `q = 2`
- `q = 4`
- `q = 6`

What we are comparing here is:

> **how the behaviour of Bayesian Optimisation changes as the number of candidate points proposed per round increases**

This is a comparison of different levels of parallel experimentation under the same optimisation problem, and it sets up the core batch-size performance analysis that follows.

In [ ]:
batch_sizes_to_compare = [1, 2, 4, 6]
batch_histories = {}

for q in batch_sizes_to_compare:
    torch.manual_seed(seed)

    train_X_q = draw_sobol_samples(bounds=bounds, n=1, q=n_init).squeeze(0)
    train_Y_q = objective_rosenbrock(train_X_q)

    history_q = run_batch_bo_loop(
        train_X_init=train_X_q,
        train_Y_init=train_Y_q,
        objective_fn=objective_rosenbrock,
        bounds=bounds,
        batch_size=q,
        n_rounds=n_rounds,
        num_restarts=20,
        raw_samples=256,
        mc_samples=128,
    )

    batch_histories[q] = history_q

## 7. Comparing batch BO performance by BO round and by total evaluations

After running batch BO under several different batch sizes, we now want to compare the resulting optimisation behaviour directly.

This figure does that in **two complementary ways**:

- the **left panel** compares performance by **BO round**
- the **right panel** compares performance by **total number of evaluations**

This distinction is extremely important in batch Bayesian Optimisation.

A larger batch size means that each BO round evaluates more points before the surrogate is updated again.
So larger batches may appear to improve more quickly **per round**, simply because they spend more evaluations in each round.

That is why both views are needed.

---

### What the code does

For each stored history in `batch_histories`, the code extracts:

```python
best_vals_q = [float(torch.min(h["train_Y"])) for h in hist]
```

This gives the best objective value found so far at each stored BO state for a given batch size `q`.

Then:

- in the **left panel**, those values are plotted against the BO round index
- in the **right panel**, the same values are plotted against:

```python
eval_counts_q = [h["train_X"].shape[0] for h in hist]
```

which is the total number of observed points available at that state

So the two panels use the same optimisation histories, but they place them on two different horizontal axes.

---

### Why `eps = 1e-6` is included

The small positive constant

```python
eps = 1e-6
```

is added to the plotted values as a numerical safeguard.

In the current version of the figure, the y-axis is not logarithmic, so this epsilon does not materially affect the shape of the curves.
It simply keeps the plotting code safe and flexible in case a log scale is used later.

So it can be thought of as a harmless numerical convenience.

---

## How to read the left panel: performance by BO round

The left panel answers the question:

> **After this many BO rounds, how good is the best solution found so far?**

This view is useful because batch BO is often motivated by **parallel experimentation**.

If a lab or simulation platform can evaluate many points at once, then one BO round may correspond to:

- one experimental campaign
- one plate of conditions
- one scheduled batch of runs
- or one cycle of parallel computations

In that sense, the left panel reflects **wall-clock decision rounds** rather than raw evaluation count.

### Interpreting the behaviour

A typical pattern in this kind of figure is:

- larger batches can improve quickly in the first few rounds
- some curves may plateau for several rounds
- later improvements can come in sudden jumps rather than smoothly

That is exactly what we should expect in BO.

A batch may fail to improve the best value for several rounds, then suddenly discover a much better region when one of the proposed batches reaches a stronger basin.

So the step-like behaviour is normal.

In the attached figure, the larger-batch runs tend to improve more rapidly in terms of **rounds**, which is intuitive:
each round gives them more chances to include at least one strong point.

So the left panel supports the practical message that:

> if experiments can truly be run in parallel, larger batches can make BO look faster when progress is measured in optimisation rounds.

---

## How to read the right panel: performance by total evaluations

The right panel answers a different question:

> **After spending this many total evaluations, how good is the best solution found so far?**

This is often the fairer comparison when thinking about **sample efficiency**.

A batch of size 6 uses evaluations much more quickly than a batch of size 1.
So even if it improves faster by round, that does not automatically mean it is more efficient per evaluation.

That is why this second panel is so important.

### Interpreting the behaviour

In many cases, the right panel shows that the differences between batch sizes become less dramatic once performance is measured against **total evaluation count** rather than rounds.

This is exactly the right interpretation.

The right panel separates two ideas that are easy to confuse:

- **round efficiency**
  how fast improvement happens per BO update cycle

- **evaluation efficiency**
  how much improvement is obtained for a given total experimental budget

So even if a large batch size looks highly attractive in the left panel, the right panel may show that the advantage is partly or mostly coming from spending evaluations faster.

That distinction is central to batch BO.

---

## Why the curves often show flat plateaus

In both panels, the curves are cumulative best-value trajectories.

That means they can only:

- stay flat
- or move downward

Flat plateaus simply mean that the latest round or the latest evaluations did not beat the current best solution.

This is completely normal in BO, especially in higher-dimensional problems.

A plateau does not mean the optimiser is broken.
It only means that the newly evaluated points did not improve the best value yet.

This can happen because:

- the surrogate is still uncertain
- the batch is exploring alternative regions
- or the optimiser is refining its understanding before eventually finding a stronger basin

---

## Why different batch sizes can behave so differently

Changing the batch size changes the BO trajectory in a fundamental way.

A larger batch size means:

- more points are chosen jointly
- more evaluations are made before the next surrogate refit
- and the surrogate is updated less frequently relative to the number of new observations

So different values of `q` do not simply scale the same optimisation path.
They create genuinely different optimisation dynamics.

This is why one batch size may look much better or worse than another in a given run.

That is not a bug.
It is part of the nature of batch BO.

---

## What this figure is really comparing

This figure is not asking:

> which batch size is universally best?

It is asking a more useful practical question:

> **How does the choice of batch size change BO behaviour depending on whether we care about optimisation rounds or total evaluation cost?**

That is the correct framing for real applications.

For example:

- if instrument time is the main bottleneck, performance by **round** may matter more
- if materials or simulation calls are the main bottleneck, performance by **total evaluations** may matter more

So the two panels correspond to two different practical notions of efficiency.

---

## Why this figure is important in the tutorial

This is one of the most important figures in the notebook, because it makes the central batch BO trade-off explicit.

Without the right panel, a reader might conclude too quickly that larger batches are simply better.
Without the left panel, a reader might miss why larger batches are attractive in the first place.

Together, the two panels show that batch BO must always be interpreted through **both**:

- parallel decision efficiency
- and total evaluation efficiency

That is one of the key conceptual lessons of the whole tutorial.

---

### Key takeaway

This figure compares batch BO behaviour under different batch sizes in two different ways:

- the **left panel** measures progress by **BO round**
- the **right panel** measures progress by **total evaluations**

The general behaviour to look for is:

- larger batches often improve faster **per round**
- but that advantage may become smaller or look different when measured **per total evaluation**

So the figure highlights one of the most important practical ideas in batch Bayesian Optimisation:

> **round efficiency and evaluation efficiency are not the same thing, and both are needed to interpret batch BO properly.**

In [ ]:
eps = 1e-6

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for q, hist in batch_histories.items():
    best_vals_q = [float(torch.min(h["train_Y"])) for h in hist]

    axes[0].plot(
        range(len(best_vals_q)),
        np.array(best_vals_q) + eps,
        "-o",
        linewidth=2.0,
        label=f"q = {q}",
    )

axes[0].set_title("Batch BO performance by BO round", fontsize=18, fontweight="bold")
axes[0].set_xlabel("BO round", fontsize=18, fontweight="bold")
axes[0].set_ylabel("Best observed value", fontsize=18, fontweight="bold")
style_ax(axes[0])
axes[0].legend(prop={"size": 10, "weight": "bold"})

for q, hist in batch_histories.items():
    best_vals_q = [float(torch.min(h["train_Y"])) for h in hist]
    eval_counts_q = [h["train_X"].shape[0] for h in hist]

    axes[1].plot(
        eval_counts_q,
        np.array(best_vals_q) + eps,
        "-o",
        linewidth=2.0,
        label=f"q = {q}",
    )

axes[1].set_title("Batch BO performance by total evaluations", fontsize=18, fontweight="bold")
axes[1].set_xlabel("Total evaluations", fontsize=18, fontweight="bold")
axes[1].set_ylabel("Best observed value", fontsize=18, fontweight="bold")
style_ax(axes[1])
axes[1].legend(prop={"size": 10, "weight": "bold"})

plt.tight_layout()
plt.show()

## 8. Comparing mean within-batch pairwise distance across batch sizes

This figure provides a simple diagnostic for how **spread out** the proposed batch points are at each BO round.

For a batch of candidate points, we compute all pairwise Euclidean distances within that batch and then take their mean.
So this quantity is a compact summary of **within-batch diversity**:

- a **larger** value means the batch points are, on average, more spread out
- a **smaller** value means the batch is more clustered

This makes the figure useful for answering a structural question that the best-value plots cannot answer directly:

> **When BO proposes a batch, how broadly is that batch distributed across the search space?**

---

### What the code does

The helper function

```python
mean_pairwise_distance(X)
```

computes the mean of all unique pairwise distances within a batch `X`.

A special case appears when the batch contains fewer than two points.
In that case there are no pairwise distances to average, so the function returns `0.0`.

This is why the `q = 1` curve lies exactly at zero in the figure:
a batch of size 1 has no within-batch spread by definition.

The code then computes this quantity for every evaluated batch in each BO history:

```python
diversities = [mean_pairwise_distance(h["candidates"]) for h in hist[:-1]]
```

and plots the resulting trajectories against BO round.

---

### How to read the figure

The horizontal axis is:

- **BO round**

The vertical axis is:

- **mean within-batch pairwise distance**

So each point answers the question:

> *At this BO round, how spread out were the points in the proposed batch, on average?*

This is not a performance measure like best observed value.
Instead, it is a **geometry-of-the-batch** measure.

So the figure should be interpreted as describing the internal structure of the proposed candidate set, not the optimisation quality directly.

---

### Why higher `q` often leads to higher mean within-batch pairwise distance

The main qualitative behaviour in this figure is that larger batch sizes tend to show **larger mean within-batch pairwise distances**.

That makes sense.

When `q` is larger, the acquisition optimiser is choosing more points jointly in one round.
A larger batch has more opportunity, and often more pressure, to distribute points across multiple promising or informative regions rather than collapsing all of them into one location.

So in general:

- `q = 1` has zero within-batch distance by definition
- `q = 2` can only measure the spacing between two points, so the result can vary substantially from round to round
- `q = 4` and `q = 6` usually have more room to form broader batches, which often leads to larger average pairwise spacing

That is the main reason the higher-`q` curves tend to sit higher.

---

### Why the `q = 2` curve can fluctuate strongly

For `q = 2`, the mean pairwise distance is just the distance between the two selected points.

So the curve can swing quite a bit depending on whether a particular round proposes:

- two fairly nearby points
- or two more separated ones

That makes the `q = 2` trajectory more sensitive and sometimes more jagged than the larger-batch curves.

By contrast, when `q = 4` or `q = 6`, the mean is averaged over many pairwise distances, so the resulting curve is often somewhat more stable.

---

### Why the `q = 1` curve is exactly zero

This is a useful sanity check.

A batch of size 1 contains only one point, so there are no pairs to compare.
That is why the helper function explicitly returns `0.0` in that case.

So the flat zero line for `q = 1` is not a bug.
It simply reflects the fact that a one-point batch has no internal spatial spread.

---

### Why this figure is useful

This figure is useful because it gives a compact summary of **how batch BO allocates its points within a round**.

The earlier best-value plots told us:

- how quickly BO improves
- and how performance looks by rounds or by total evaluations

But they did not tell us whether the proposed batches are:

- tightly clustered
- moderately spread out
- or strongly diversified

This figure helps fill that gap.

So it complements the performance plots by showing something about the **structure of the batch itself**.

---

### Is this the only possible diversity metric?

No.

This metric is useful, but it is not unique.

For example, one could also examine:

- the **minimum** within-batch pairwise distance
- the **maximum** within-batch pairwise distance
- coordinate-wise spread
- or visual projections of the batch candidates

Those alternatives can sometimes be more informative for specific questions.

For example:

- minimum distance is especially relevant when a hard exclusion radius is being enforced
- maximum distance is especially relevant for inclusion-type clustering rules

So the mean pairwise distance is best understood as a **simple summary statistic**, not a complete description of batch geometry.

Still, for the present comparison across standard batch sizes, it is a reasonable and interpretable first diagnostic.

---

### What this suggests about batch BO behaviour

The overall qualitative lesson from this figure is:

> **larger batch sizes often produce batches that are more spatially spread out on average.**

That does not automatically mean larger batches are better.
But it does help explain why larger batches may behave differently from smaller ones.

A more spread-out batch may:

- cover multiple promising regions in one round
- provide broader information to the surrogate
- and potentially improve faster per round

At the same time, it may also mean that the batch is less locally concentrated around one promising basin.

So within-batch spread is part of the broader exploration–exploitation trade-off in batch BO.

---

### Key takeaway

This figure compares the **mean within-batch pairwise distance** across different batch sizes.

Its main message is that:

- `q = 1` has zero within-batch spread by definition
- and, in general, **higher batch sizes tend to produce larger average pairwise distances within the batch**

So the figure helps show that larger batches are often not just “more points,” but also **more spatially distributed sets of points** within each BO round.

In [ ]:
def mean_pairwise_distance(X):
    if X.shape[0] < 2:
        return 0.0
    dists = torch.cdist(X, X, p=2)
    upper = dists[torch.triu(torch.ones_like(dists), diagonal=1) > 0]
    return float(torch.mean(upper))


batch_diversity = {}
for q, hist in batch_histories.items():
    diversities = [mean_pairwise_distance(h["candidates"]) for h in hist[:-1]]
    batch_diversity[q] = diversities

fig, ax = plt.subplots(figsize=(8, 5))

for q, diversities in batch_diversity.items():
    ax.plot(
        range(len(diversities)),
        diversities,
        "-o",
        linewidth=2.0,
        label=f"q = {q}",
    )

ax.set_title("Mean within-batch pairwise distance", fontsize=18, fontweight="bold")
ax.set_xlabel("BO round", fontsize=18, fontweight="bold")
ax.set_ylabel("Mean pairwise distance", fontsize=18, fontweight="bold")
style_ax(ax)
ax.legend(prop={"size": 10, "weight": "bold"})

plt.tight_layout()
plt.show()

## 9. Introducing explicit control over batch scattering and clustering

So far, the notebook has focused on **standard batch BO**, where the acquisition function itself decides how spread out or how concentrated the points in each batch should be.

That is often a sensible default.

However, in many real optimisation settings, we may want more direct control over the **internal geometry of the batch**.

For example, we might prefer a batch in which the points are:

- deliberately **spread out**, so that one round covers a broader region of the search space
- or deliberately **clustered**, so that one round performs more local refinement around a promising region

This cell introduces a simple way to impose that kind of control explicitly.

---

### The core idea

The main idea is to add a **distance-based acceptance rule** when constructing the batch.

Instead of always accepting whatever candidate is returned by acquisition optimisation, we now ask whether that candidate is sufficiently compatible with the points already selected into the same batch.

This gives us a way to bias the batch towards either:

- **scattering**
- or **clustering**

by changing a simple rule.

---

## What the code introduces

This cell defines three related functions:

1. `is_valid_candidate(...)`
2. `select_batch_with_radius_rule(...)`
3. `run_batch_bo_loop_with_radius_rule(...)`

Together, these form a modified batch BO workflow in which the spatial structure of the batch can be controlled explicitly.

---

## 1. `is_valid_candidate(...)`: the local acceptance rule

The function

```python
is_valid_candidate(...)
```

decides whether a newly proposed point should be accepted into the current batch.

It computes the distances from the new candidate to the points already selected in that batch:

```python
dists = torch.cdist(candidate, selected_tensor)
```

and then uses those distances differently depending on the chosen `mode`.

---

### Exclusion mode: encourage scattering

When

```python
mode == "exclude"
```

the candidate is accepted only if its **minimum distance** to the already selected points is at least the chosen radius:

```python
min_dist >= radius
```

This means:

> every new point must stay at least `radius` away from the points already in the batch.

So exclusion mode pushes the batch towards being **more spread out**.

This is a direct way to encourage batch diversity.

---

### Inclusion mode: encourage clustering

When

```python
mode == "include"
```

the candidate is accepted only if:

- its **maximum distance** to the already selected points is no larger than the chosen radius
- and it is not an exact duplicate, enforced through `min_sep`

So the inclusion rule says:

> every new point must remain within a local neighbourhood of the current batch.

This encourages the batch to stay **more clustered**.

So, conceptually:

- exclusion mode promotes **scattering**
- inclusion mode promotes **clustering**

This is the main new idea introduced in this cell.

---

## 2. `select_batch_with_radius_rule(...)`: building the batch under that rule

The next function,

```python
select_batch_with_radius_rule(...)
```

uses `is_valid_candidate(...)` to build an entire batch one point at a time.

At a high level, it works as follows:

1. fit the acquisition function in the usual way
2. propose one candidate at a time using single-point acquisition optimisation
3. test whether that candidate satisfies the chosen radius rule relative to the points already selected
4. accept or reject accordingly
5. repeat until the batch is full, or until no more valid points can be found

If repeated acquisition optimisation fails to produce a valid point, the code falls back to random Sobol sampling and applies the same validity rule there as well.

So the batch is still built from BO-guided candidate proposals, but now filtered through an explicit spatial rule.

---

### Why there is a feasibility check for exclusion mode

The code computes:

```python
max_possible_distance = torch.norm(bounds[1] - bounds[0], p=2).item()
```

and checks whether the chosen exclusion radius is even possible in the domain.

This is necessary because if the exclusion radius is larger than the largest possible pairwise distance in the search box, then no valid batch can ever be formed.

So this is a useful sanity check for the scattering case.

---

## 3. `run_batch_bo_loop_with_radius_rule(...)`: running a full BO loop with explicit batch-shape control

The final function wraps this radius-based batch selection into a full BO loop:

```python
run_batch_bo_loop_with_radius_rule(...)
```

This is structurally very similar to the earlier `run_batch_bo_loop(...)`, except that instead of selecting the batch directly through standard `qLogExpectedImprovement`, it now builds the batch through the custom radius-rule selector.

So the BO workflow remains familiar:

1. fit a GP surrogate
2. define the acquisition logic
3. construct the batch
4. evaluate the batch
5. update the dataset
6. store the history

The main difference is that the **internal geometry of the batch is now being controlled deliberately**.

---

## Why this is useful

This gives us a new kind of experimental control.

Instead of treating batch geometry as an emergent property of the acquisition function, we now introduce a simple mechanism for steering it.

That means the tutorial can now study three different styles of batch construction:

- **standard joint batch BO**, where the acquisition function decides the batch shape
- **exclusion-style batch construction**, which pushes the batch to be more spread out
- **inclusion-style batch construction**, which pushes the batch to be more clustered

This makes the exploration–exploitation trade-off much more concrete.

---

## Why the clustering case is the “trickier” one

The scattering case is comparatively intuitive:

- if we require points to stay far apart, the batch becomes more spread out

The clustering case is more subtle.

At first glance, one might think that controlling clustering should be just as easy as controlling scattering, simply by reversing the inequality.
But in practice, clustering is **trickier**.

Why?

Because the candidate proposals are still coming from **global acquisition optimisation**.

That means the optimiser is naturally trying to find strong points anywhere in the domain.
If we then try to force all selected points to remain close together, we are effectively asking a global acquisition rule to behave like a local refinement rule.

Those two objectives do not always align naturally.

So clustering is not just the “opposite” of scattering in a practical sense.
It is often harder to control cleanly because the acquisition generator itself is still global.

That is why the inclusion mode should be interpreted as a useful **heuristic mechanism for encouraging clustering**, rather than a perfect local batch-construction method.

This is the “trick” part:

> **making points stay far apart is straightforward; making them stay meaningfully close together while still behaving like good BO candidates is more delicate.**

That is an important practical lesson.

---

## What the parameters mean

### `mode`
Controls whether the batch is encouraged to be:

- `"exclude"` → more scattered
- `"include"` → more clustered

### `radius`
Sets the scale of that rule.

- in exclusion mode, larger radius means stronger spreading
- in inclusion mode, smaller radius means stronger clustering

So the meaning of the radius depends on the mode.

### `min_sep`
Used in inclusion mode to avoid accepting exact duplicates or numerically identical points.

This is important because otherwise a clustering rule could collapse into selecting the same point repeatedly.

---

## Why this section matters in the tutorial

This section is important because it moves beyond comparing standard batch sizes and begins to ask a deeper question:

> **Can we deliberately shape the internal structure of the batch itself?**

That is a more advanced and more practical question.

In real applications, it may be desirable to control whether a batch behaves more like:

- a diverse exploration set
- or a tightly focused local refinement set

So this section helps connect BO more directly to how an experimentalist or practitioner might think about a batch of runs.

---

### Key takeaway

This cell introduces an explicit distance-based mechanism for controlling the **scattering or clustering behaviour** of batch BO proposals.

It does this by:

- checking whether each new point satisfies either an **exclusion** rule or an **inclusion** rule relative to the already selected points
- using that rule to construct the batch one point at a time
- and embedding the resulting batch selector into a full BO loop

The key practical idea is:

> **batch geometry can be influenced directly, not only indirectly through the acquisition function.**

At the same time, the clustering case is the trickier one, because enforcing local concentration while still using globally proposed BO candidates is inherently more delicate than simply enforcing separation.

In [ ]:
def is_valid_candidate(candidate, selected_tensor, mode="exclude", radius=0.5, min_sep=1e-3, tol=1e-8):
    if selected_tensor is None:
        return True

    dists = torch.cdist(candidate, selected_tensor)
    min_dist = torch.min(dists).item()
    max_dist = torch.max(dists).item()

    if mode == "exclude":
        return bool(min_dist >= radius - tol)

    if mode == "include":
        return bool((max_dist <= radius + tol) and (min_dist >= min_sep))

    raise ValueError("mode must be 'exclude' or 'include'")


def select_batch_with_radius_rule(
    model,
    train_Y,
    bounds,
    batch_size=4,
    mode="exclude",
    radius=0.45,
    num_restarts=20,
    raw_samples=256,
    max_attempts_per_point=25,
    max_fallback_attempts=200,
):
    selected_tensor = None

    max_possible_distance = torch.norm(bounds[1] - bounds[0], p=2).item()
    if mode == "exclude" and radius > max_possible_distance:
        raise ValueError(
            f"radius={radius} exceeds the maximum possible distance in the domain ({max_possible_distance:.3f})."
        )

    best_f = (-train_Y).max().item()
    acq = LogExpectedImprovement(model=model, best_f=best_f)

    for point_idx in range(batch_size):
        accepted = False

        for _ in range(max_attempts_per_point):
            candidate, _ = optimize_acqf(
                acq_function=acq,
                bounds=bounds,
                q=1,
                num_restarts=num_restarts,
                raw_samples=raw_samples,
            )
            if is_valid_candidate(candidate, selected_tensor, mode=mode, radius=radius):
                selected_tensor = candidate if selected_tensor is None else torch.cat([selected_tensor, candidate], dim=0)
                accepted = True
                break

        if not accepted:
            for _ in range(max_fallback_attempts):
                fallback = draw_sobol_samples(bounds=bounds, n=1, q=1).reshape(1, -1)
                if is_valid_candidate(fallback, selected_tensor, mode=mode, radius=radius):
                    selected_tensor = fallback if selected_tensor is None else torch.cat([selected_tensor, fallback], dim=0)
                    accepted = True
                    break

        if not accepted:
            print(
                f"Warning: could not place point {point_idx + 1}/{batch_size} under "
                f"mode='{mode}' with radius={radius}. Returning a smaller batch of size "
                f"{0 if selected_tensor is None else selected_tensor.shape[0]}."
            )
            break

    if selected_tensor is None:
        raise RuntimeError("Failed to construct any valid batch candidates.")

    return selected_tensor


def run_batch_bo_loop_with_radius_rule(
    train_X_init,
    train_Y_init,
    objective_fn,
    bounds,
    batch_size=4,
    n_rounds=8,
    mode="exclude",
    radius=0.45,
    num_restarts=20,
    raw_samples=256,
):
    train_X = train_X_init.clone()
    train_Y = train_Y_init.clone()
    history = []

    for round_idx in range(n_rounds + 1):
        train_Y_bo = -train_Y
        gp = SingleTaskGP(
            train_X=train_X,
            train_Y=train_Y_bo,
            input_transform=Normalize(d=train_X.shape[-1]),
            outcome_transform=Standardize(m=1),
        )
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll(mll)
        gp.eval()

        candidates = select_batch_with_radius_rule(
            model=gp,
            train_Y=train_Y,
            bounds=bounds,
            batch_size=batch_size,
            mode=mode,
            radius=radius,
            num_restarts=num_restarts,
            raw_samples=raw_samples,
        )

        best_idx = torch.argmin(train_Y)
        history.append({
            "round": round_idx,
            "train_X": train_X.clone(),
            "train_Y": train_Y.clone(),
            "candidates": candidates.clone(),
            "batch_size": batch_size,
            "mode": mode,
            "radius": radius,
            "best_x": train_X[best_idx].clone(),
            "best_y": train_Y[best_idx].clone(),
            "lengthscale": gp.covar_module.lengthscale.detach().cpu().reshape(-1).numpy().copy(),
            "noise": gp.likelihood.noise.detach().cpu().item(),
        })

        if round_idx < n_rounds:
            train_X = torch.cat([train_X, candidates], dim=0)
            train_Y = torch.cat([train_Y, objective_fn(candidates)], dim=0)

    return history

## 10. Running three batch-construction strategies for direct comparison

We now set up a direct comparison between three different ways of constructing a batch of size 4.

This is an important transition in the notebook.

Earlier, we compared different values of `q` under **standard batch BO**, where the acquisition function itself determines the internal geometry of the batch.
Here, we keep the batch size fixed at:

```python
q = 4
```

and instead compare **how the batch is constructed**.

So the question changes from:

> **What happens when the number of points in the batch changes?**

to:

> **What happens when the rule governing the spatial structure of the batch changes?**

That is exactly what this cell is designed to test.

---

### What the code does

The cell runs three BO workflows:

1. **Standard batch BO**
2. **Exclusion-radius batch BO**
3. **Inclusion-radius batch BO**

Each run starts from a freshly generated initial Sobol design, with the random seed reset beforehand so that the comparison remains reproducible.

At the end, the three histories are stored as:

- `history_standard_q4`
- `history_exclusion_q4`
- `history_inclusion_q4`

So this cell is mainly a setup cell for the diagnostic plots that follow.

---

## 1. Standard batch BO

The first run is:

```python
history_standard_q4 = run_batch_bo_loop(...)
```

This is the reference case.

Here, the batch is selected through the standard joint batch BO mechanism using the acquisition function directly.
No additional geometric rule is imposed on how close or far apart the points in the batch should be.

So this run answers:

> **What kind of batch does standard BO propose if we simply let the acquisition function decide?**

This will serve as the baseline against which the more explicitly controlled strategies are compared.

---

## 2. Exclusion-radius batch BO

The second run is:

```python
history_exclusion_q4 = run_batch_bo_loop_with_radius_rule(
    ...
    mode="exclude",
    radius=2.0,
)
```

This uses the same BO problem and the same batch size, but now each new point is accepted into the batch only if it stays at least a distance of `2.0` away from the points already selected in that batch.

So the exclusion rule is designed to encourage **scattering**:

- the batch should cover a broader region
- near-duplicate points are discouraged
- and the batch is pushed toward greater internal diversity

This makes the exclusion run a simple way to impose a stronger exploration-style geometry on the batch.

---

## 3. Inclusion-radius batch BO

The third run is:

```python
history_inclusion_q4 = run_batch_bo_loop_with_radius_rule(
    ...
    mode="include",
    radius=2.0,
)
```

This is the opposite geometric bias.

Now a new point is accepted only if it stays within a distance of `2.0` from the already selected points, while also respecting the small minimum-separation rule introduced earlier to prevent exact duplicates.

So the inclusion rule is designed to encourage **clustering**:

- the batch should stay more local
- the points should be closer together
- and the batch should behave more like a local refinement set

This gives a useful contrast to the exclusion-radius construction.

---

### Why the seed is reset before each run

Before each of the three runs, the code calls:

```python
torch.manual_seed(seed)
```

This is done so that each comparison starts from a controlled and reproducible random state.

That is important because otherwise differences between the three runs could be partly caused by unrelated randomness in the initial design rather than by the batch-construction strategy itself.

So the seed reset helps make the comparison cleaner and fairer.

It does **not** make the runs identical, because once the batch-selection rule changes, the whole BO trajectory changes.
But it does ensure that the setup is reproducible and that the comparison begins from a consistent random state.

---

### Why a fresh initial design is regenerated for each strategy

Each strategy regenerates its own initial dataset:

```python
train_X_compare
train_X_compare_excl
train_X_compare_incl
```

and similarly for `train_Y`.

This is a reasonable design for two reasons:

1. each run is kept self-contained
2. each radius-rule experiment begins from a clean BO state rather than reusing a partially evolved trajectory

Because the seed is reset before each draw, these initial datasets are still directly comparable.

So the point here is not to compare three arbitrary unrelated BO runs, but to compare three reproducible workflows that differ primarily in how the batch itself is built.

---

### What is being held fixed

Across the three runs, all of the following remain fixed:

- the objective function
- the search bounds
- the input dimension
- the initial design size
- the batch size
- the number of BO rounds
- and the acquisition-optimisation settings

So the point of the comparison is **not** to change the optimisation problem.
It is to isolate the effect of the **batch-construction rule**.

That is what makes the later comparisons meaningful.

---

### What is actually being compared

This cell compares three distinct philosophies of batch construction:

#### Standard BO
Let the acquisition function determine the batch naturally.

#### Exclusion-radius BO
Force the batch points to be more spread out.

#### Inclusion-radius BO
Force the batch points to remain more local and clustered.

So this is a comparison not of different objectives, but of different **geometric biases** inside the same batch BO framework.

That makes it a very useful experiment, because it helps us understand whether performance differences are related not only to how many points are in the batch, but also to **how those points are arranged relative to each other**.

---

### Why this comparison is useful

This is one of the most practically meaningful comparisons in the tutorial.

In real optimisation workflows, a batch is not just an abstract mathematical object.
It may correspond to:

- several experiments run together
- several simulations submitted at once
- several conditions loaded into a plate
- or several opportunities to use instrument time in parallel

In those settings, we may care not only about the number of runs in the batch, but also about whether the batch is:

- broad and diverse
- or concentrated and locally refined

So this cell helps move the notebook from:

- standard batch-size comparison
to
- explicit control of batch geometry

That is a more advanced and more realistic question.

---

### Why the inclusion case is particularly interesting

The inclusion case is especially important because it highlights a subtle point:

> making the batch more clustered is not just the simple mirror image of making it more spread out.

In practice, clustering is harder to control cleanly because the candidate generator is still based on global acquisition optimisation.
So the inclusion-radius strategy should be interpreted as a useful **heuristic for encouraging locality**, rather than as a perfect local-search mechanism.

That is part of what makes this comparison worth studying.

---

### What this cell prepares us to examine next

Once these three histories have been created, the notebook can compare them using several diagnostics, such as:

- best observed value over BO rounds
- mean within-batch pairwise distance
- minimum or maximum pairwise distance under the chosen rule
- and the overall effect of scattering versus clustering on optimisation behaviour

So this cell is the setup step for one of the most conceptually interesting parts of the tutorial.

---

### Key takeaway

This cell runs three batch BO workflows with the same batch size `q = 4`, but with three different batch-construction strategies:

- **standard** batch BO
- **exclusion-radius** batch BO, which encourages scattering
- **inclusion-radius** batch BO, which encourages clustering

So the main comparison here is:

> **how the optimisation behaviour changes when we explicitly control the internal geometry of the batch, rather than simply letting the acquisition function decide it.**

In [ ]:
torch.manual_seed(seed)

train_X_compare = draw_sobol_samples(bounds=bounds, n=1, q=n_init).squeeze(0)
train_Y_compare = objective_rosenbrock(train_X_compare)

history_standard_q4 = run_batch_bo_loop(
    train_X_init=train_X_compare,
    train_Y_init=train_Y_compare,
    objective_fn=objective_rosenbrock,
    bounds=bounds,
    batch_size=4,
    n_rounds=n_rounds,
    num_restarts=20,
    raw_samples=256,
    mc_samples=128,
)

torch.manual_seed(seed)

train_X_compare_excl = draw_sobol_samples(bounds=bounds, n=1, q=n_init).squeeze(0)
train_Y_compare_excl = objective_rosenbrock(train_X_compare_excl)

history_exclusion_q4 = run_batch_bo_loop_with_radius_rule(
    train_X_init=train_X_compare_excl,
    train_Y_init=train_Y_compare_excl,
    objective_fn=objective_rosenbrock,
    bounds=bounds,
    batch_size=4,
    n_rounds=8,
    mode="exclude",
    radius=2.0,
    num_restarts=20,
    raw_samples=256,
)

torch.manual_seed(seed)

train_X_compare_incl = draw_sobol_samples(bounds=bounds, n=1, q=n_init).squeeze(0)
train_Y_compare_incl = objective_rosenbrock(train_X_compare_incl)

history_inclusion_q4 = run_batch_bo_loop_with_radius_rule(
    train_X_init=train_X_compare_incl,
    train_Y_init=train_Y_compare_incl,
    objective_fn=objective_rosenbrock,
    bounds=bounds,
    batch_size=4,
    n_rounds=8,
    mode="include",
    radius=2.0,
    num_restarts=20,
    raw_samples=256,
)


## 11. Comparing standard, exclusion-radius, and inclusion-radius batch BO

We now compare the three batch-construction strategies introduced in the previous cell:

- **standard batch BO**
- **exclusion-radius batch BO**
- **inclusion-radius batch BO**

This figure is designed to answer two different questions at once:

1. **How do these strategies differ in optimisation performance?**
2. **How do these strategies differ in the geometry of the batches they propose?**

That is why the figure is split into two panels.

---

### What the code does

The first three lines extract the cumulative best observed value from each BO history:

```python
standard_best = [float(torch.min(h["train_Y"])) for h in history_standard_q4]
exclusion_best = [float(torch.min(h["train_Y"])) for h in history_exclusion_q4]
inclusion_best = [float(torch.min(h["train_Y"])) for h in history_inclusion_q4]
```

So these three sequences track the best objective value found so far over BO rounds for each batch-construction strategy.

The next three lines compute the mean within-batch pairwise distance for the evaluated candidate batches:

```python
standard_diversity = [mean_pairwise_distance(h["candidates"]) for h in history_standard_q4[:-1]]
exclusion_diversity = [mean_pairwise_distance(h["candidates"]) for h in history_exclusion_q4[:-1]]
inclusion_diversity = [mean_pairwise_distance(h["candidates"]) for h in history_inclusion_q4[:-1]]
```

The slice `[:-1]` is used because, as before, the final stored history entry contains a proposed but not evaluated batch.

So the figure combines:

- **left panel:** optimisation performance
- **right panel:** average within-batch spread

This is a very useful pairing, because it compares both the **outcome** and the **internal geometry** of the three batch strategies.

---

## How to read the left panel

The left panel plots the **best observed value by BO round**.

This is the most direct optimisation metric.

Each curve answers the question:

> *After this many BO rounds, what is the lowest objective value found so far?*

Because this is a minimisation problem:

- lower is better
- plateaus mean recent batches did not improve the current best solution
- downward jumps mean a newly evaluated batch discovered a better point

So the left panel compares how quickly and how strongly the three batch-construction rules improve the optimisation.

---

## How to read the right panel

The right panel plots the **mean within-batch pairwise distance**.

This is not a performance metric.
Instead, it is a simple summary of how spread out the points inside each proposed batch are.

So each point answers the question:

> *At this BO round, how far apart were the points in the batch, on average?*

This lets us compare the internal geometry of the three batch strategies:

- whether the batch is naturally concentrated
- deliberately scattered
- or deliberately clustered

So the right panel helps explain *why* the optimisation behaviour in the left panel may differ.

---

## What the comparison is trying to show conceptually

This is not just a comparison of three curves.

It is really a comparison of three different ways of allocating a fixed batch of 4 evaluations:

### 1. Standard batch BO
The acquisition function decides the batch geometry naturally.

### 2. Exclusion-radius BO
The batch is encouraged to be more spread out.

### 3. Inclusion-radius BO
The batch is encouraged to remain more local and clustered.

So the figure is comparing different **geometric biases** imposed on the same BO problem.

That is what makes it so interesting.

---

## General interpretation of the expected behaviour

Although the exact numerical results may depend on the seed and the specific optimisation trajectory, there are some general qualitative patterns we would expect.

### Standard BO
The standard batch often sits somewhere in the middle.
It is free to produce batches that are either somewhat concentrated or somewhat diverse, depending on what the acquisition function thinks is valuable.

So its within-batch distance is usually neither maximally large nor maximally small.

### Exclusion-radius BO
The exclusion rule is designed to force the batch points apart.
So we would generally expect this curve to show **larger mean within-batch pairwise distances** than the standard case.

That is the intended effect of the exclusion-radius rule.

### Inclusion-radius BO
The inclusion rule is designed to keep points closer together.
So we would generally expect the inclusion curve to lie **below** the standard and exclusion curves in the right panel, reflecting a more clustered batch.

This is the intended local-refinement bias of the inclusion rule.

So in the right panel, the qualitative ordering often reflects exactly the geometry that each strategy is trying to encourage.

---

## Why the left-panel performance differences matter

The left panel then asks whether those different batch geometries lead to different optimisation outcomes.

This is where the real trade-off appears.

### A more spread-out batch may help because:
- it covers more of the search space in one round
- it reduces redundancy within the batch
- it may discover a better basin more quickly

### A more clustered batch may help because:
- it performs stronger local refinement
- it focuses more evaluations near one promising region
- it can behave more like a local optimisation step inside BO

So neither strategy is automatically superior.
The question is whether the particular problem benefits more from:

- broader within-round exploration
- or tighter within-round exploitation

That is exactly what the left panel helps us examine.

---

## Why this figure is especially useful

This figure is useful because it links:

- **batch geometry**
- and **optimisation performance**

in the same visual comparison.

Without the right panel, we would only know that one strategy performed better or worse.
Without the left panel, we would only know that the batches were more or less spread out.

Together, the two panels let us ask:

> **Did changing the internal spacing of the batch actually change optimisation behaviour in a meaningful way?**

That is the key question of this section.

---

## Why this matters in real applications

This comparison is not just an artificial notebook exercise.

In real experimental settings, a batch might represent:

- several conditions on a reaction plate
- several simulations launched together
- several materials formulations tested in one round
- or several instrument slots used in parallel

In those contexts, a researcher may genuinely want to choose between:

- a more diverse batch that probes several regions at once
- and a more clustered batch that refines one promising region

So this figure connects directly to a realistic design choice in applied BO.

---

## One important caveat

The inclusion-radius strategy is more heuristic than the exclusion case.

That is because clustering is harder to control cleanly while still using globally proposed BO candidates.

So the inclusion results should be interpreted as:

- a useful illustration of clustering bias
- not a perfect or unique implementation of local batch refinement

That does not reduce the value of the comparison, but it is worth keeping in mind.

---

### Key takeaway

This figure compares three different batch-construction strategies:

- standard batch BO
- exclusion-radius batch BO
- inclusion-radius batch BO

The **left panel** shows how they differ in optimisation progress, while the **right panel** shows how they differ in the average spread of the points inside each batch.

Taken together, the figure helps show that:

> **changing the internal geometry of the batch can change not only how the batch looks, but also how the optimisation behaves.**

This makes the exploration–exploitation trade-off in batch BO much more concrete.

In [ ]:
standard_best = [float(torch.min(h["train_Y"])) for h in history_standard_q4]
exclusion_best = [float(torch.min(h["train_Y"])) for h in history_exclusion_q4]
inclusion_best = [float(torch.min(h["train_Y"])) for h in history_inclusion_q4]

standard_diversity = [mean_pairwise_distance(h["candidates"]) for h in history_standard_q4[:-1]]
exclusion_diversity = [mean_pairwise_distance(h["candidates"]) for h in history_exclusion_q4[:-1]]
inclusion_diversity = [mean_pairwise_distance(h["candidates"]) for h in history_inclusion_q4[:-1]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    range(len(standard_best)),
    np.array(standard_best) + 1e-6,
    "-o",
    linewidth=2.0,
    label="Standard q=4",
)
axes[0].plot(
    range(len(exclusion_best)),
    np.array(exclusion_best) + 1e-6,
    "-o",
    linewidth=2.0,
    label="Exclusion radius q=4",
)
axes[0].plot(
    range(len(inclusion_best)),
    np.array(inclusion_best) + 1e-6,
    "-o",
    linewidth=2.0,
    label="Inclusion radius q=4",
)
axes[0].set_title("Best observed value by BO round", fontsize=18, fontweight="bold")
axes[0].set_xlabel("BO round", fontsize=18, fontweight="bold")
axes[0].set_ylabel("Best observed value", fontsize=18, fontweight="bold")
style_ax(axes[0])
axes[0].legend(prop={"size": 9, "weight": "bold"})

axes[1].plot(
    range(len(standard_diversity)),
    standard_diversity,
    "-o",
    linewidth=2.0,
    label="Standard q=4",
)
axes[1].plot(
    range(len(exclusion_diversity)),
    exclusion_diversity,
    "-o",
    linewidth=2.0,
    label="Exclusion radius q=4",
)
axes[1].plot(
    range(len(inclusion_diversity)),
    inclusion_diversity,
    "-o",
    linewidth=2.0,
    label="Inclusion radius q=4",
)
axes[1].set_title("Mean within-batch pairwise distance", fontsize=18, fontweight="bold")
axes[1].set_xlabel("BO round", fontsize=18, fontweight="bold")
axes[1].set_ylabel("Mean pairwise distance", fontsize=18, fontweight="bold")
style_ax(axes[1])
axes[1].legend(prop={"size": 9, "weight": "bold"})

plt.tight_layout()
plt.show()

## 12. Checking whether the radius rules are actually being enforced

The previous figure compared the three batch-construction strategies using the **mean within-batch pairwise distance**.
That was useful as a general summary of batch spread, but it did not directly test whether the explicit radius rules were being satisfied.

This cell does that more directly.

Instead of using the mean pairwise distance for all three cases, it uses the statistic that is most relevant to each rule:

- for **standard batch BO**, we plot the **minimum** within-batch pairwise distance as a reference
- for **exclusion-radius BO**, we also plot the **minimum** within-batch pairwise distance, because the exclusion rule is supposed to keep this quantity **above** the target radius
- for **inclusion-radius BO**, we plot the **maximum** within-batch pairwise distance, because the inclusion rule is supposed to keep this quantity **below** the target radius

So this figure is a more direct diagnostic of whether the batch-spacing rules are doing what they were designed to do.

---

### What the code does

The helper functions

```python
min_pairwise_distance(X)
max_pairwise_distance(X)
```

compute the smallest and largest Euclidean distance between any two points in a batch.

These are then applied to the stored candidate batches:

```python
standard_min_dists
exclusion_min_dists
inclusion_max_dists
```

So the three plotted curves correspond to:

- **standard q=4** → minimum pairwise distance
- **exclusion q=4** → minimum pairwise distance
- **inclusion q=4** → maximum pairwise distance

A dashed horizontal line is also added at:

```python
y = 2.0
```

to show the chosen target radius.

This makes the figure easy to interpret visually.

---

## How to read the figure

The horizontal axis is:

- **BO round**

The vertical axis is:

- the relevant pairwise-distance statistic for each strategy

So the interpretation of the curves is:

### Standard q=4
This is just a reference curve.
It shows the minimum pairwise distance that standard batch BO naturally produces, without any explicit spacing rule.

### Exclusion q=4
This curve should stay **at or above** the target radius if the exclusion rule is doing its job.

### Inclusion q=4
This curve should stay **at or below** the target radius if the inclusion rule is doing its job.

So the dashed line is the key reference for interpreting whether the radius constraint is being respected.

---

## Why the inclusion curve is very close to 2.0

The most striking feature of the plot is that the inclusion curve lies very close to the dashed line at `2.0`.

This makes good sense.

Recall that the inclusion rule accepts a new point only if its distance to the already selected points remains within the inclusion radius:

```python
max_dist <= radius
```

with:

```python
radius = 2.0
```

So the inclusion strategy is actively constructing the batch under a rule that tries to keep the **largest pairwise distance** no greater than `2.0`.

That means the most relevant quantity to inspect is exactly the one being plotted here:

> the **maximum within-batch pairwise distance**

Since the algorithm is repeatedly trying to fill the batch while staying within that radius, it is natural that the resulting maximum distance often ends up **very close to the boundary value itself**.

In other words, the inclusion rule tends to produce batches that are:

- as spread out as possible
- while still remaining inside the allowed clustering radius

So the inclusion curve hovering near `2.0` is not suspicious.
It is exactly what we should expect from a boundary-constrained selection rule.

---

### Why it is close to the boundary rather than much smaller

This is an important subtlety.

The inclusion rule does **not** try to make the batch as tight as possible.
It only requires that the batch stay **within** the allowed radius.

Meanwhile, the candidate proposals are still being driven by global acquisition optimisation, which is trying to find strong BO candidates rather than minimise within-batch distance.

So the optimiser will often accept points that are:

- still legal under the inclusion rule
- but as far apart as the rule allows

That is why the maximum pairwise distance often lands near the boundary.

So the inclusion rule behaves less like:

> “make the points very close together”

and more like:

> “do not let the points spread beyond this limit”

That is why the curve sits near `2.0` rather than much lower.

---

## Why the exclusion curve can sit above 2.0

The exclusion rule requires:

```python
min_dist >= radius
```

So the smallest pairwise distance in the batch should remain at or above `2.0`.

That is exactly the relevant diagnostic for the exclusion case.

The exclusion curve often sits **above** the boundary rather than exactly on it because once the batch is forced to spread out, the resulting minimum pairwise distance may naturally exceed the threshold by a substantial margin.

So in the exclusion case, the radius is acting as a lower bound rather than a target value.

That is why the orange curve is often comfortably above `2.0`.

---

## Why the standard curve can fall well below 2.0

The standard batch BO run has no explicit radius rule.
So its minimum pairwise distance can move freely according to whatever the acquisition function decides is useful.

That is why the standard curve can drop well below the dashed line:
there is nothing in the standard batch-construction mechanism forcing the points to maintain any minimum separation.

So the blue curve serves as a useful baseline showing what the batch geometry looks like without explicit spacing control.

---

## Why this figure is important

This figure is important because it validates the radius-rule construction much more directly than the mean-distance plot.

The mean pairwise distance is useful as a summary, but it does not directly answer questions like:

- “is the exclusion rule really keeping points at least this far apart?”
- “is the inclusion rule really keeping the batch inside the chosen radius?”

This plot answers those questions much more directly.

So it acts as a **diagnostic check** that the custom batch-construction rules are behaving as intended.

---

## What this tells us about batch design

This figure reinforces an important practical idea:

- the **exclusion rule** creates a batch with a guaranteed minimum level of separation
- the **inclusion rule** creates a batch with a controlled upper bound on spread

So the two rules shape the batch in different geometric ways:

- exclusion creates a floor on pairwise spacing
- inclusion creates a ceiling on pairwise spacing

That is exactly why different distance statistics are needed to evaluate them properly.

---

### Key takeaway

This figure checks the spacing rules more directly by plotting:

- the **minimum** within-batch pairwise distance for standard and exclusion-radius BO
- and the **maximum** within-batch pairwise distance for inclusion-radius BO

The fact that the inclusion curve stays very close to `2.0` is expected, because the inclusion rule is designed to keep the batch **within** that radius, and the candidate selection process tends to use as much of that allowed spread as possible while still satisfying the constraint.

So this figure confirms that the radius-based batch-construction rules are not just changing the average spread of the batch, but are also enforcing the intended geometric bounds in a more direct sense.

In [ ]:
def min_pairwise_distance(X):
    if X.shape[0] < 2:
        return 0.0
    dists = torch.cdist(X, X, p=2)
    upper = dists[torch.triu(torch.ones_like(dists), diagonal=1) > 0]
    return float(torch.min(upper))

def max_pairwise_distance(X):
    if X.shape[0] < 2:
        return 0.0
    dists = torch.cdist(X, X, p=2)
    upper = dists[torch.triu(torch.ones_like(dists), diagonal=1) > 0]
    return float(torch.max(upper))

standard_min_dists = [min_pairwise_distance(h["candidates"]) for h in history_standard_q4[:-1]]
exclusion_min_dists = [min_pairwise_distance(h["candidates"]) for h in history_exclusion_q4[:-1]]
inclusion_max_dists = [max_pairwise_distance(h["candidates"]) for h in history_inclusion_q4[:-1]]

fig, ax = plt.subplots(figsize=(8, 5))


ax.plot(
    range(len(standard_min_dists)),
    standard_min_dists,
    "-o",
    linewidth=2.0,
    label="Standard q=4 (min dist)",
)
ax.plot(
    range(len(exclusion_min_dists)),
    exclusion_min_dists,
    "-o",
    linewidth=2.0,
    label="Exclusion q=4 (min dist)",
)
ax.plot(
    range(len(inclusion_max_dists)),
    inclusion_max_dists,
    "-o",
    linewidth=2.0,
    label="Inclusion q=4 (max dist)",
)
ax.axhline(
    y=2.0,
    linestyle="--",
    linewidth=2.0,
    color="green",
    label="Target radius = 2.0",
)

ax.set_title("Radius-rule diagnostics for batch spacing", fontsize=18, fontweight="bold")
ax.set_xlabel("BO round", fontsize=18, fontweight="bold")
ax.set_ylabel("Pairwise distances", fontsize=18, fontweight="bold")
style_ax(ax)
ax.legend(prop={"size": 10, "weight": "bold"})

plt.tight_layout()
plt.show()

## 🧭 Closing Remarks

In this tutorial, we moved from **high-dimensional Bayesian Optimisation with one candidate set at a time** to the problem of running BO when evaluations can be performed in **parallel batches**.

The central idea was that once multiple experiments or simulations can be launched together, the BO problem changes in an important way.

The challenge is no longer only:

- how to model the objective,
- how to define an acquisition function,
- and how to repeat the BO loop,

but also:

- how to choose **multiple points jointly** rather than one point at a time,
- how to interpret progress when one BO round may contain several evaluations,
- how to compare performance fairly across different batch sizes,
- and how to think about the **internal geometry of the batch itself**.

At a structural level, the BO workflow still followed the same general pattern:

1. fit a Gaussian Process surrogate to the currently observed data,
2. build an acquisition rule from that surrogate,
3. choose the next evaluation set,
4. evaluate the objective at those points,
5. update the dataset,
6. and repeat.

But the meaning of one optimisation step changed.

In standard sequential BO, one step means one new point.

Here, one BO round can mean:

- two new points,
- four new points,
- six new points,
- or another chosen batch size.

That shift was the whole point of the notebook.

Instead of treating BO as a process that always makes one decision at a time, we treated it as something that must still work when the decision unit becomes a **set of points**.

That is much closer to many real experimental and computational workflows.

Across the notebook, we used the same dimension-customisable Rosenbrock setting to make this shift concrete.

That continuity was useful, because it meant the new ideas could be understood without simultaneously changing the underlying optimisation problem.

What changed was not the benchmark itself, but the **way BO interacted with it**.

The first important lesson was that batch BO is not just sequential BO with a different tensor shape.

Changing from `q = 1` to `q > 1` did not merely change how many points were returned by `optimize_acqf`.
It changed the structure of the BO decision problem itself.

A batch is not just several independent next points.
It is a **joint proposal**.

That is why this tutorial introduced:

- `qLogExpectedImprovement`,
- Monte Carlo batch acquisition evaluation,
- batch-shaped candidate tensors,
- and dataset updates that append several new observations at once.

So one of the main conceptual takeaways was that:

> batch BO changes the unit of decision from “the next point” to “the next set of points.”

The notebook then showed that this has immediate consequences for how BO performance should be interpreted.

A larger batch size may appear to improve much faster when plotted against **BO round**, because each round contains more evaluations.

But that same apparent advantage can look smaller, or at least different, when performance is plotted against **total evaluations**.

That led to one of the most important practical lessons of the notebook:

> round efficiency and evaluation efficiency are not the same thing.

That distinction is central in batch BO.

If a researcher is limited mainly by:

- experimental cycles,
- instrument scheduling,
- or wall-clock turnaround,

then performance by BO round may be the most relevant view.

If the real constraint is instead:

- material usage,
- simulation cost,
- total number of reactions,
- or total measurement budget,

then performance by total evaluations may matter more.

So the batch-size comparison did not just show which curve looked better.
It showed that the answer depends on **what kind of cost we care about**.

The notebook also highlighted that batch BO is not only about how many points are proposed, but also about **how those points are arranged inside the batch**.

The coordinate plots and mean within-batch pairwise distance diagnostics showed that larger batches are often more spatially distributed.

That made the internal structure of a batch visible rather than treating the batch as a black box.

This was an important practical step, because in real applications a batch may correspond to:

- several experiments on one plate,
- several simulations submitted together,
- several synthesis conditions prepared in parallel,
- or multiple instrument runs performed in one cycle.

In that setting, the batch is not just a mathematical object.
It is a practical experimental plan.

That is why the later part of the notebook moved beyond standard batch BO and introduced explicit control over **batch geometry**.

By imposing:

- an **exclusion radius**, we encouraged the batch to become more spread out
- an **inclusion radius**, we encouraged the batch to remain more clustered

This made the exploration–exploitation trade-off much more concrete.

Instead of only discussing that trade-off abstractly, we were able to shape it directly through the batch-construction rule.

That was one of the most interesting practical lessons of the notebook:

> in batch BO, we can sometimes influence not only *which* points are selected, but also *how the selected points relate to one another geometrically*.

At the same time, the notebook also made clear that controlling clustering is the trickier case.

Making points stay far apart is conceptually straightforward.
Making them stay meaningfully close together while still behaving like strong BO candidates is more delicate, because the candidate generator is still driven by a global acquisition rule.

So the inclusion-radius strategy was useful not because it gave a perfect local-search mechanism, but because it exposed an important practical difficulty:

> encouraging local refinement inside a globally proposed batch is harder than simply enforcing separation.

This is exactly the kind of subtlety that starts to matter once BO is used in realistic workflows rather than toy examples.

That is where this notebook connects to real research.

In a real laboratory or simulation campaign, batch BO may be used when:

- multiple reaction conditions can be tested simultaneously,
- several catalysts can be prepared in one synthesis cycle,
- multiple process conditions can be evaluated in parallel,
- or many simulations can be launched before waiting for results.

In those settings, the researcher is often not asking only:

> what is the next best point?

but rather:

> what is the next **set** of experiments worth spending one full round of effort on?

That is a fundamentally batch-BO question.

So by the end of this notebook, we have not introduced a completely different optimisation paradigm.
Instead, we have taken the BO workflow already built earlier and adapted it to a much more realistic mode of use:

- multiple evaluations per round,
- multiple ways to judge efficiency,
- and explicit control over whether the batch behaves more like a diverse exploration set or a clustered refinement set.

That gives us a natural stopping point:

> we now know how to build, run, and interpret a practical batch Bayesian Optimisation workflow, and how to think critically about both the size and the geometry of the batch.

The next stage will naturally ask what happens when the batch BO setting becomes even more realistic:

- when variables are mixed or constrained,
- when evaluation budgets are highly structured,
- when BO must operate under practical workflow restrictions,
- or when human decision-making is part of the loop.

That is where batch BO starts to move from a useful algorithmic extension to a genuinely practical optimisation framework.